# Chapter 9: Smoothing and Perplexity

## 1. Nhập thư viện và định nghĩa mô hình N-gram

In [3]:
from collections import Counter
import json
from math import exp, log
from pathlib import Path
import re

WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_DIRECTORY = next((directory for directory in (WORKING_DIRECTORY, *WORKING_DIRECTORY.parents) if (directory / 'data').is_dir()), None)
if PROJECT_DIRECTORY is None:
    raise FileNotFoundError('Could not locate the project data directory.')
DATASET_PATH = PROJECT_DIRECTORY / 'data' / 'c4-train.00000-of-01024-30K.json' / 'c4-train.00000-of-01024-30K.json'
DOCUMENT_LIMIT = 10_000
PREDICTION_LIMIT = 5
TRAIN_SHARE = 0.80
VALIDATION_SHARE = 0.10
START_TOKEN = '<s>'
END_TOKEN = '</s>'
TOKEN_PATTERN = re.compile(r"[a-z]+(?:'[a-z]+)?")
SENTENCE_PATTERN = re.compile(r'[.!?]+(?:\s+|$)')
MODEL_ORDERS = {'unigram': 1, 'bigram': 2, 'trigram': 3}
PREDICTION_CONTEXTS = ('the cat', 'natural language', 'machine learning', 'data science', 'artificial intelligence')
RANKING_CONTEXT = 'machine learning'
RANKING_CANDIDATES = ('is useful for natural language processing', 'banana computer quickly', 'studies language models')

def tokenize(text):
    """Convert text to lowercase word tokens."""
    return TOKEN_PATTERN.findall(text.lower())


def load_sentences(path, limit):
    """Load tokenized sentences from the JSONL corpus."""
    sentences = []
    with path.open(encoding='utf-8') as corpus_file:
        for line in corpus_file:
            text = json.loads(line).get('text', '')
            sentences.extend(tokens for sentence in SENTENCE_PATTERN.split(text) if (tokens := tokenize(sentence)))
            if len(sentences) >= limit:
                break
    return sentences


def split_corpus(sentences):
    """Split sentences into deterministic train, validation, and test partitions."""
    training_end = int(len(sentences) * TRAIN_SHARE)
    validation_end = training_end + int(len(sentences) * VALIDATION_SHARE)
    return sentences[:training_end], sentences[training_end:validation_end], sentences[validation_end:]


class NGramLanguageModel:
    """Count-based N-gram language model with optional Laplace smoothing."""

    def __init__(self, order, laplace):
        """Initialize model configuration and count tables."""
        self.order, self.laplace = order, laplace
        self.ngram_counts, self.context_counts = Counter(), Counter()
        self.vocabulary = {END_TOKEN}

    def fit(self, sentences):
        """Fit the model on tokenized sentences."""
        context_size = self.order - 1
        for sentence in sentences:
            padded = [START_TOKEN] * context_size + list(sentence) + [END_TOKEN]
            self.vocabulary.update(sentence)
            for index in range(context_size, len(padded)):
                context, word = tuple(padded[index - context_size:index]), padded[index]
                self.ngram_counts[context + (word,)] += 1
                self.context_counts[context] += 1
        return self

    def probability(self, context, word):
        """Return the conditional probability of a word."""
        context_size = self.order - 1
        context = tuple(context[-context_size:]) if context_size else ()
        count, total = self.ngram_counts[context + (word,)], self.context_counts[context]
        return (count + 1) / (total + len(self.vocabulary)) if self.laplace else (count / total if total else 0.0)

    def perplexity(self, sentences):
        """Calculate perplexity over tokenized sentences."""
        context_size, total_log, token_count = self.order - 1, 0.0, 0
        for sentence in sentences:
            padded = [START_TOKEN] * context_size + list(sentence) + [END_TOKEN]
            for index in range(context_size, len(padded)):
                probability = self.probability(padded[index - context_size:index], padded[index])
                if not probability:
                    return float('inf')
                total_log, token_count = total_log + log(probability), token_count + 1
        return exp(-total_log / token_count)

    def predict_next(self, context_text, limit):
        """Return the most probable next words for a text context."""
        context = tokenize(context_text)
        probabilities = ((word, self.probability(context, word)) for word in self.vocabulary)
        return sorted(probabilities, key=lambda item: item[1], reverse=True)[:limit]

    def rank_sentences(self, context_text, candidates):
        """Rank continuations by conditional log probability."""
        rankings = []
        for candidate in candidates:
            history, score = tokenize(context_text), 0.0
            for word in tokenize(candidate) + [END_TOKEN]:
                probability = self.probability(history, word)
                score, history = score + (log(probability) if probability else float('-inf')), history + [word]
            rankings.append((candidate, score))
        return sorted(rankings, key=lambda item: item[1], reverse=True)


## 2. Tải dữ liệu và chạy đánh giá

In [4]:
sentences = load_sentences(DATASET_PATH, DOCUMENT_LIMIT)
training_corpus, validation_corpus, test_corpus = split_corpus(sentences)
result_rows = []

for model_name, order in MODEL_ORDERS.items():
    for smoothing_name, laplace in (('mle', False), ('laplace', True)):
        model = NGramLanguageModel(order, laplace).fit(training_corpus)
        for split_name, corpus in (('train', training_corpus), ('validation', validation_corpus), ('test', test_corpus)):
            result_rows.append({'record_type': 'perplexity', 'model': model_name, 'smoothing': smoothing_name, 'split': split_name, 'perplexity': model.perplexity(corpus)})
        for context in PREDICTION_CONTEXTS:
            for rank, (word, probability) in enumerate(model.predict_next(context, PREDICTION_LIMIT), start=1):
                result_rows.append({'record_type': 'prediction', 'model': model_name, 'smoothing': smoothing_name, 'context': context, 'candidate': word, 'rank': rank, 'probability': probability})
        for rank, (candidate, score) in enumerate(model.rank_sentences(RANKING_CONTEXT, RANKING_CANDIDATES), start=1):
            result_rows.append({'record_type': 'ranking', 'model': model_name, 'smoothing': smoothing_name, 'context': RANKING_CONTEXT, 'candidate': candidate, 'rank': rank, 'value': score})


## 3. Xem kết quả perplexity

In [5]:
[row for row in result_rows if row['record_type'] == 'perplexity']


[{'record_type': 'perplexity',
  'model': 'unigram',
  'smoothing': 'mle',
  'split': 'train',
  'perplexity': 1108.820002105717},
 {'record_type': 'perplexity',
  'model': 'unigram',
  'smoothing': 'mle',
  'split': 'validation',
  'perplexity': inf},
 {'record_type': 'perplexity',
  'model': 'unigram',
  'smoothing': 'mle',
  'split': 'test',
  'perplexity': inf},
 {'record_type': 'perplexity',
  'model': 'unigram',
  'smoothing': 'laplace',
  'split': 'train',
  'perplexity': 1125.879489046683},
 {'record_type': 'perplexity',
  'model': 'unigram',
  'smoothing': 'laplace',
  'split': 'validation',
  'perplexity': 1268.7149243078134},
 {'record_type': 'perplexity',
  'model': 'unigram',
  'smoothing': 'laplace',
  'split': 'test',
  'perplexity': 1637.9587229952629},
 {'record_type': 'perplexity',
  'model': 'bigram',
  'smoothing': 'mle',
  'split': 'train',
  'perplexity': 38.36975647102643},
 {'record_type': 'perplexity',
  'model': 'bigram',
  'smoothing': 'mle',
  'split': 'vali